In [2]:
import pandas as pd
from pathlib import Path
import os

# Change to project root
if Path.cwd().name == "tft":
    os.chdir("../..")
elif Path.cwd().name == "notebooks":
    os.chdir("..")

p = Path("data/processed/pretraining/germany/global/tft_inputs/regional_train_tft_full.parquet")
df = pd.read_parquet(p, columns=["plant_id","power_norm","plant_01","plant_02","plant_03","plant_05","plant_06"])

print(df[["plant_01","plant_02","plant_03","plant_05","plant_06"]].describe())
print(df["plant_id"].value_counts())
print("max onehot sum per row:", (df[["plant_01","plant_02","plant_03","plant_05","plant_06"]].sum(axis=1)).max())


            plant_01       plant_02       plant_03       plant_05  \
count  142190.000000  142190.000000  142190.000000  142190.000000   
mean        0.224172       0.224172       0.224854       0.101948   
std         0.417037       0.417037       0.417488       0.302581   
min         0.000000       0.000000       0.000000       0.000000   
25%         0.000000       0.000000       0.000000       0.000000   
50%         0.000000       0.000000       0.000000       0.000000   
75%         0.000000       0.000000       0.000000       0.000000   
max         1.000000       1.000000       1.000000       1.000000   

            plant_06  
count  142190.000000  
mean        0.224854  
std         0.417488  
min         0.000000  
25%         0.000000  
50%         0.000000  
75%         0.000000  
max         1.000000  
plant_id
plant_03    31972
plant_06    31972
plant_01    31875
plant_02    31875
plant_05    14496
Name: count, dtype: int64
max onehot sum per row: 1.0


In [3]:
import pyarrow.parquet as pq

p = "data/processed/pretraining/germany/global/tft_inputs/regional_train_tft_full.parquet"
pf = pq.ParquetFile(p)
print("rows:", pf.metadata.num_rows)
print("cols:", pf.schema.names[:40], "... total", len(pf.schema.names))


rows: 142190
cols: ['timestamp_utc', 'power_norm', 'poa_irradiance', 'plant_id', 'plant_01', 'plant_02', 'plant_03', 'plant_05', 'plant_06', 'lstm_enc_000', 'lstm_enc_001', 'lstm_enc_002', 'lstm_enc_003', 'lstm_enc_004', 'lstm_enc_005', 'lstm_enc_006', 'lstm_enc_007', 'lstm_enc_008', 'lstm_enc_009', 'lstm_enc_010', 'lstm_enc_011', 'lstm_enc_012', 'lstm_enc_013', 'lstm_enc_014', 'lstm_enc_015', 'lstm_enc_016', 'lstm_enc_017', 'lstm_enc_018', 'lstm_enc_019', 'lstm_enc_020', 'lstm_enc_021', 'lstm_enc_022', 'lstm_enc_023', 'lstm_enc_024', 'lstm_enc_025', 'lstm_enc_026', 'lstm_enc_027', 'lstm_enc_028', 'lstm_enc_029', 'lstm_enc_030'] ... total 97


In [5]:
import duckdb
p="data/processed/pretraining/germany/global/tft_inputs/regional_train_tft_full.parquet"
con=duckdb.connect()
print(con.execute(f"select count(*) n_rows from read_parquet('{p}')").fetchall())
print(con.execute(f"select plant_id, count(*) n from read_parquet('{p}') group by plant_id order by n desc").df())
print(con.execute(f"select min(power_norm), max(power_norm) from read_parquet('{p}')").fetchall())
print(con.execute(f"select * from read_parquet('{p}') limit 5").df())



[(142190,)]
   plant_id      n
0  plant_06  31972
1  plant_03  31972
2  plant_02  31875
3  plant_01  31875
4  plant_05  14496
[(0.0, 1.0)]
              timestamp_utc  power_norm  poa_irradiance  plant_id  plant_01  \
0 2023-01-01 23:00:00+00:00         0.0       -0.671514  plant_01       1.0   
1 2023-01-01 23:15:00+00:00         0.0       -0.671514  plant_01       1.0   
2 2023-01-01 23:30:00+00:00         0.0       -0.671514  plant_01       1.0   
3 2023-01-01 23:45:00+00:00         0.0       -0.671514  plant_01       1.0   
4 2023-01-02 00:00:00+00:00         0.0       -0.671514  plant_01       1.0   

   plant_02  plant_03  plant_05  plant_06  lstm_enc_000  ...  \
0       0.0       0.0       0.0       0.0      0.013960  ...   
1       0.0       0.0       0.0       0.0      0.013925  ...   
2       0.0       0.0       0.0       0.0      0.013895  ...   
3       0.0       0.0       0.0       0.0      0.013898  ...   
4       0.0       0.0       0.0       0.0      0.013900  ...   

 

In [6]:
import pandas as pd
from pathlib import Path

train = Path("data/processed/pretraining/germany/global/tft_inputs/regional_train_tft_full.parquet")
val   = Path("data/processed/pretraining/germany/global/tft_inputs/regional_val_tft_full.parquet")

for p in [train, val]:
    df = pd.read_parquet(p)
    print("\n==", p.name, "==")
    print("shape:", df.shape)
    print("first 30 cols:", df.columns[:30].tolist())
    print("last 30 cols:", df.columns[-30:].tolist())
    print("\nALL COLS:")
    print("\n".join(df.columns.tolist()))


== regional_train_tft_full.parquet ==
shape: (142190, 97)
first 30 cols: ['timestamp_utc', 'power_norm', 'poa_irradiance', 'plant_id', 'plant_01', 'plant_02', 'plant_03', 'plant_05', 'plant_06', 'lstm_enc_000', 'lstm_enc_001', 'lstm_enc_002', 'lstm_enc_003', 'lstm_enc_004', 'lstm_enc_005', 'lstm_enc_006', 'lstm_enc_007', 'lstm_enc_008', 'lstm_enc_009', 'lstm_enc_010', 'lstm_enc_011', 'lstm_enc_012', 'lstm_enc_013', 'lstm_enc_014', 'lstm_enc_015', 'lstm_enc_016', 'lstm_enc_017', 'lstm_enc_018', 'lstm_enc_019', 'lstm_enc_020']
last 30 cols: ['lstm_enc_058', 'lstm_enc_059', 'lstm_enc_060', 'lstm_enc_061', 'lstm_enc_062', 'lstm_enc_063', 'global_tilted_irradiance_instant_raw', 'direct_normal_irradiance_instant_raw', 'shortwave_radiation_instant_raw', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'weather_code', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m', 'shortwave_radiation_instant', 'direct_radiation_instant', 'diffuse_radiation_instant', 'direct_normal_irradianc

In [7]:
import pandas as pd
import numpy as np

P = "data/processed/pretraining/germany/global/tft_inputs/regional_train_tft_full.parquet"  # adjust
df = pd.read_parquet(P)

a = df["poa_irradiance"].astype(float)

cands = [
    "global_tilted_irradiance_instant_raw",
    "global_tilted_irradiance_instant",
    "pvlib_poa_global",
]

for c in cands:
    if c not in df.columns:
        print("missing:", c)
        continue
    b = df[c].astype(float)

    ok = np.isfinite(a) & np.isfinite(b)
    corr = np.corrcoef(a[ok], b[ok])[0,1]
    max_abs = np.max(np.abs((a[ok] - b[ok]).to_numpy()))
    print(f"{c:35s} corr={corr:.6f} max_abs_diff={max_abs:.6f} n={ok.sum()}")


global_tilted_irradiance_instant_raw corr=1.000000 max_abs_diff=736.167630 n=142190
global_tilted_irradiance_instant    corr=1.000000 max_abs_diff=736.167630 n=142190
pvlib_poa_global                    corr=0.916747 max_abs_diff=999.976809 n=142190


In [8]:
import pandas as pd
import numpy as np

P = "data/processed/pretraining/germany/global/tft_inputs/regional_train_tft_full.parquet"  # adjust
df = pd.read_parquet(P)

a = df["poa_irradiance"].astype(float)
cands = [
    "global_tilted_irradiance_instant_raw",
    "global_tilted_irradiance_instant",
    "pvlib_poa_global",
]

print("poa_irradiance stats:", a.min(), a.max(), a.mean(), a.std())

for c in cands:
    if c not in df.columns:
        print("missing:", c)
        continue
    b = df[c].astype(float)

    ok = np.isfinite(a) & np.isfinite(b)
    corr = np.corrcoef(a[ok], b[ok])[0, 1]
    max_abs = np.max(np.abs((a[ok] - b[ok]).to_numpy()))
    print(f"{c:35s} corr={corr:.6f} max_abs_diff={max_abs:.6f} n={ok.sum()}")


poa_irradiance stats: -0.671514157251241 3.762909963602529 0.000802496292274071 1.0001921434999073
global_tilted_irradiance_instant_raw corr=1.000000 max_abs_diff=736.167630 n=142190
global_tilted_irradiance_instant    corr=1.000000 max_abs_diff=736.167630 n=142190
pvlib_poa_global                    corr=0.916747 max_abs_diff=999.976809 n=142190
